## Assignment 3, Question 7: Valuation of Asian Options via Monte Carlo Simulation

Import relevant libraries

In [5]:
import numpy as np

Setup

In [6]:
# Random seed for reproducibility
np.random.seed(42)

# Simulation parameters
S_0 = 100  # Initial stock price
K = 100  # Strike price
mu = 0.15  # Mean return
sigma = 0.2  # Volatility
r = 0.08 # Risk-free rate
T = 1  # Time to maturity (1 year)
delta_t = 1/250 # Time step (daily)
n_steps = int(T / delta_t) # Number of time steps

# One master matrix of normal deviates
deviates_master = np.random.normal(0, 1, (n_steps, 1000))

Helper functions

In [7]:
def simulate_gbm(S_0: float, r: float, sigma: float, delta_t: float, deviates: np.ndarray) -> np.ndarray:
    """
    Simulate N paths of Geometric Brownian Motion (GBM) stock prices.

    Args:
    - S_0: Initial stock price
    - r: Risk-free rate
    - sigma: Volatility
    - delta_t: Time step (in years)
    - deviates: A numpy array of shape (n_steps, N) containing the normal deviates for the simulation

    Returns:
    - A numpy array of shape (n_steps + 1, N) containing the simulated GBM paths.
    """

    n_steps, N = deviates.shape

    S = np.zeros((n_steps + 1, N))
    S[0, :] = S_0
    for t in range(1, n_steps + 1):
        S[t, :] = S[t-1, :] * np.exp((r - 0.5 * sigma**2) * delta_t + sigma * np.sqrt(delta_t) * deviates[t-1, :])

    return S  

def option_value_se_ci(S: np.ndarray, antithetic: bool = False) -> dict:
    """
    Calculate the option value, standard error, and confidence interval from simulated GBM paths.

    Args:
    - S: A numpy array of shape (n_steps + 1, N) containing the simulated GBM paths.
    - antithetic: Whether the paths were generated using the antithetic variable technique (default: False).

    Returns:
    - A dictionary containing the option value, standard error, and confidence interval.
    """

    res = {}

    # Option payoff at maturity (by path)
    V_T = np.maximum(S[1:, :].mean(axis = 0) - K, 0)

    # Discounted payoff by path
    discounted_V_T = np.exp(-r * T) * V_T

    if antithetic:
        m = discounted_V_T.shape[0] // 2
        estimator_samples = 0.5 * (discounted_V_T[:m] + discounted_V_T[m:])
    else:
        estimator_samples = discounted_V_T

    # Option value (discounted expected payoff)
    res["option_value"] = estimator_samples.mean()

    # Standard error of the option value
    sigma_V = estimator_samples.std(ddof=1)  # Sample standard deviation
    res["standard_error"] = sigma_V / np.sqrt(len(estimator_samples))

    # 95% confidence interval
    z_score = 1.96  
    res["confidence_interval"] = (
        res["option_value"] - z_score * res["standard_error"],
        res["option_value"] + z_score * res["standard_error"]
    )

    return res 


Run computation

In [8]:
config = {
    "a": {
        "N": 5, 
        "antithetic": False,
        "label": "Part (a): N = 5 sample paths"
    },
    "b": {
        "N": 10, 
        "antithetic": False,
        "label": "Part (b): N = 10 sample paths"
    },
    "c": {
        "N": 20, 
        "antithetic": False,
        "label": "Part (c): N = 20 sample paths"
    },
    "d": {
        "N": 10, 
        "antithetic": True,
        "label": "Part (d): N = 10 sample paths with variance reduction"
    },
    "e": {
        "N": 20, 
        "antithetic": True,
        "label": "Part (e): N = 20 sample paths with variance reduction"
    },
    "f": {
        "N": 100, 
        "antithetic": False,
        "label": "Part (f): N = 100  sample paths"
    },
    "g": {
        "N": 500, 
        "antithetic": False,
        "label": "Part (g): N = 500  sample paths"
    },
    "h": {
        "N": 1000, 
        "antithetic": False,
        "label": "Part (h): N = 1000  sample paths"
    },
    "i": {
        "N": 500, 
        "antithetic": True,
        "label": "Part (i): N = 500  sample paths with variance reduction"
    },
        "j": {
        "N": 1000, 
        "antithetic": True,
        "label": "Part (j): N = 1000  sample paths with variance reduction"
    },
}

for part, params in config.items():
    print("-" * 80)
    print(params["label"])
    print("-" * 80)
    print()

    N = params["N"]
    antithetic = params["antithetic"]

    if antithetic:
        # Use the first N/2 columns of the master deviates for the original paths, 
        # And concatenate with their antithetic counterparts to get N paths in total
        deviates_base = deviates_master[:, :N//2]
        deviates = np.concatenate((deviates_base, -deviates_base), axis=1)
    else:
        deviates = deviates_master[:, :N]

    S = simulate_gbm(
        S_0, r, sigma, delta_t, deviates
    )
    res = option_value_se_ci(S, antithetic=antithetic)
    print(f"Estimated option value: {res['option_value']:.4f}")
    print(f"Standard error: {res['standard_error']:.4f}")
    print(f"95% confidence interval: ({res['confidence_interval'][0]:.4f}, {res['confidence_interval'][1]:.4f})")
    print()
    print("-" * 80)


--------------------------------------------------------------------------------
Part (a): N = 5 sample paths
--------------------------------------------------------------------------------

Estimated option value: 11.4287
Standard error: 1.5695
95% confidence interval: (8.3524, 14.5051)

--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
Part (b): N = 10 sample paths
--------------------------------------------------------------------------------

Estimated option value: 10.7881
Standard error: 2.1163
95% confidence interval: (6.6401, 14.9361)

--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
Part (c): N = 20 sample paths
--------------------------------------------------------------------------------

Estimated option value: 8.6190
Standard error: 1.6167
95% conf